# ISE A3 Repository Investigation

This executable notebook verifies the frozen inputs, reproduces the clean offline build/test/start path, and inspects the recorded observations. Run from the package root or the notebook directory with CPython 3.12.14 on Linux x86_64. The included wheels do not target native Windows. No student data or external services are used.

The distributed execution was produced by executing these Python cells sequentially in one namespace. No separate Jupyter-kernel run is claimed. Rerunning creates a new workspace and preserves previous evidence.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json, subprocess, sys
ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / 'upstream/UPSTREAM.json').exists())
upstream = json.loads((ROOT / 'upstream/UPSTREAM.json').read_text())
print('Repository:', upstream['repository_url'])
print('Frozen commit:', upstream['commit'])
print('Current Python:', sys.version.split()[0])


Repository: https://github.com/pimutils/todoman
Frozen commit: 78e35788606a0add0c4f99ceb702c3ea0d0b1346
Current Python: 3.12.14


In [2]:
verified = subprocess.run([sys.executable, str(ROOT / 'scripts/reproduce.py'), '--verify-only'], cwd=ROOT, capture_output=True, text=True)
print(verified.stdout)
if verified.returncode:
    print(verified.stderr)
verified.check_returncode()


Verified 33 frozen input checksums.



In [3]:
RUN = ROOT / 'runs' / ('notebook-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
execution = subprocess.run([sys.executable, str(ROOT / 'scripts/reproduce.py'), '--output', str(RUN)], cwd=ROOT, capture_output=True, text=True)
print(execution.stdout)
if execution.returncode:
    print(execution.stderr)
execution.check_returncode()


Verified 33 frozen input checksums.
01-venv: exit 0
02-install-lock: exit 0
03-build-wheel: exit 0
04-install-wheel: exit 0
05-upstream-tests: exit 0
06-start-version: exit 0
07-study-probes: exit 0
08-dependency-check: exit 0
{
  "upstream_commit": "78e35788606a0add0c4f99ceb702c3ea0d0b1346",
  "upstream_source_sha256": "00e147fee9b31de498a08778aeec9d83e66ca99712b426a7de8132f1b8c56089",
  "python": "3.12.14",
  "platform": "Linux-6.18.55-x86_64-with-glibc2.39",
  "runtime_differs_from_recorded_python": false,
  "upstream_tests": {
    "passed": 188,
    "skipped": 6,
    "xfailed": 3,
    "failed": 0,
    "errors": 0,
    "total": 197
  },
  "study_checks_passed": 6,
  "built_wheel_sha256": "d1f8a709102190677aa9251a6132e1adeb986bb547cb5c9051fb0117b636433d",
  "offline_install": true,
  "seed": 20261025,
  "note": "Skips and expected failures are retained; characterization checks include missing safeguards."
}



## Interpret outcomes

Skipped and expected failing upstream cases remain limitations. A successful characterization check can establish a missing protection. P3 is a controlled stale-object interleaving, P4 demonstrates date-only semantics, and P5 executes a harmless marker from trusted-local Python configuration. None establishes a production security guarantee.


In [4]:
result = json.loads((RUN / 'summary.json').read_text())
checks = json.loads((RUN / 'workflow/results.json').read_text())
assert result['upstream_tests']['failed'] == 0
assert result['upstream_tests']['errors'] == 0
assert result['study_checks_passed'] == 6
print('Upstream outcomes:', result['upstream_tests'])
for item in checks['results']:
    print(item['id'], item['observation'])
print('Proposed boundary: protect the A2 service record and export only approved revisions; see ADR-001.')


Upstream outcomes: {'passed': 188, 'skipped': 6, 'xfailed': 3, 'failed': 0, 'errors': 0, 'total': 197}
P1 Normal CLI edit persists due value and increments SEQUENCE; a new CLI process and SQLite cache agree.
P2 Malformed date returns exit 2 and leaves the calendar file byte-identical.
P3 A stale Todo object overwrites a newer due date without a version-conflict error.
P4 A date-only due value is accepted and persisted as an all-day date, without a UTC instant.
P5 Configuration is executable Python; a harmless synthetic marker is written during load_config.
P6 An explicitly missing configuration produces exit 1 and actionable error text.
Proposed boundary: protect the A2 service record and export only approved revisions; see ADR-001.
